# WS 3 — Dimensionality Reduction & Unsupervised Learning

**Machine Learning in Computational Physics** · University of Vienna

---

<div class="alert alert-block alert-danger">

**Assignment submission**

When you submit this notebook for assignment, make sure that all tasks below (blue boxes) are completed (withc ode) and all questions are answered (with extra markdown responses). Make sure that the notebook runs correctly when all cells are executed in the right order from top to bottom. Before submission, do not clear outputs. Leave all the outputs from the last run included.
</div>

## 1. Motivation and Data Preparation

In this workshop, we look at dimensionality reduction and clustering techniques. Can we come up with representations that clearly resolve the structural differences in the data?

We will use the data and representations from WS2

In [ ]:
#basic stuff
import os
import sys
from functools import partial
import numpy as np
import random

import ase
from ase.io import read, write
from ase.visualize import view
from ase.build import molecule
from ase.db import connect

from matplotlib import pyplot as plt
import matplotlib as mpl
import pandas
import seaborn as sns
from weas_widget import WeasWidget

#ML stuff
import dscribe
import sklearn
from sklearn.metrics.pairwise import pairwise_kernels
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm # progress bars for loops

#acepotentials
import tempfile
from juliacall import Main as jl
jl.seval('import Pkg; Pkg.add("ACEpotentials")')
jl.seval('import Pkg; Pkg.add("ExtXYZ")')
%matplotlib inline

Preparing cyclohexane data

In [ ]:
# read in the frames from each MD simulation
traj = []
names = ['chair', 'twist-boat', 'boat', 'half-chair', 'planar']
rgb_colors = [(0.13333333333333333, 0.47058823529411764, 0.7098039215686275),
                (0.4588235294117647, 0.7568627450980392, 0.34901960784313724),
                (0.803921568627451, 0.6078431372549019, 0.16862745098039217),
                (0.803921568627451, 0.13725490196078433, 0.15294117647058825),
                (0.4392156862745098, 0.2784313725490196, 0.611764705882353),]

ranges = np.zeros((len(names), 2), dtype=int)
conf_idx = np.zeros(len(names), dtype=int)

for i, n in enumerate(names):
    # here comes the parsing of the files
    frames = read(f'./data/cyclohexane_data/MD/{n}.xyz', '::') #creates a list of Atoms objects.

    for frame in frames:
        # wrap each frame in its box, this means that we ensure that the coordinates of each atom are between 0 and the box size, which is important for the featurization step later on.
        frame.wrap(eps=1E-10)

        # mask each frame so that descriptors are only centered on carbon (#6) atoms
        mask = np.zeros(len(frame))
        mask[np.where(frame.numbers == 6)[0]] = 1 #the mask is list of 0s and 1s, where 1 indicates that the atom is a carbon atom and 0 indicates that it is not. This is used to tell the featurizer which atoms to center the descriptors on.
        frame.arrays['center_atoms_mask'] = mask

    #now we create three lists that contain the whole dataset. 
    
    # ranges is a list of tuples that indicate the start and end index of each trajectory in the traj list and traj is a list of all frames from all trajectories.
    ranges[i] = (len(traj), len(traj) + len(frames)) #list of data ranges

    conf_idx[i] = len(traj) # handy list to indicate the index of the first frame for each trajectory

    traj = [*traj, *frames] # full list of frames, concatenated to a list with 50000 entries

Preparing QM7 data

In [ ]:
def get_qm7_energies(filename, key="dft"):
    """ Returns a dictionary with heats of formation for each xyz-file.
    """

    f = open(filename, "r")
    lines = f.readlines()
    f.close()

    energies = []

    for line in lines:
        tokens = line.split()

        xyz_name = tokens[0]
        hof = float(tokens[1])
        dftb = float(tokens[2])

        if key=="delta":
            energies.append(hof - dftb)
        else:
            energies.append(hof)

    return energies

# Import QM7, already parsed to QML
from ase.io import read

qm7_dft_energy = get_qm7_energies("./data/qm7/hof_qm7.txt", key="dft")
qm7_delta_energy = get_qm7_energies("./data/qm7/hof_qm7.txt", key = "delta")

qm7 = read(f'./data/qm7/qm7.xyz', '::') #creates a list of Atoms objects.


## 2. Feature Generation

Let's generate a range of features as we did in WS2. We will use global molecular features

### MBTR

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of dscribe with Py>=3.10
from dscribe.descriptors import MBTR

min_dist = 0.8  #smallest distance in Angstrom
max_dist = 6.0  #largest distance in Angstrom
#we are constructing MBTR with inverse distances
grid_min = 1.0/max_dist # 1/Angstrom
grid_max = 1.0/min_dist # 1/Angstrom
grid_n = 100 # number of grid points
sigma = 0.05

#k=1, 1-body
#"atomic_number": The atomic number.
#k=2, 2-body
#"distance": Pairwise distance in angstroms.
#"inverse_distance": Pairwise inverse distance in 1/angstrom.
#k=3, 3-body
#"angle": Angle in degrees.
#"cosine": Cosine of the angle.

# MBTR setup for cyclohexane dataset
mbtr = MBTR(
    species=["H", "C"],
    geometry={"function": "inverse_distance"}, # "replace inverse_distance" with "cosine" to get 3-body terms instead of 2-body terms"
    grid={"min": grid_min, "max": grid_max, "n": grid_n, "sigma": sigma},
    weighting={"function": "exp", "scale": 0.5, "threshold": 1e-3},
    periodic=False,
    normalization="l2",
    sparse=False
)

mbtrs = mbtr.create(traj) #calculate MBTR for all frames


### SOAP

In [ ]:
import collections.abc
collections.Iterable = collections.abc.Iterable # this is unfortunately necessary for compatbility of describe with Py>=3.10
from dscribe.descriptors import SOAP

species = ["H", "C"]
r_cut = 3.5 #cutoff in Angstrom
n_max = 4 # 
sigma = 0.3 #stdev of gaussians
l_max = 4

# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
    average='inner', #this extra keyword makes it a global descriptor by averaging over all atoms in the molecule
)

soaps_low = soap.create(traj) #calculate SOAP for all frames

species = ["H", "C"]
r_cut = 5 #cutoff in Angstrom
n_max = 6 # 
sigma = 0.3 #stdev of gaussians
l_max = 6

# Setting up the SOAP descriptor
soap = SOAP(
    species=species,
    periodic=False,
    r_cut=r_cut,
    n_max=n_max,
    l_max=l_max,
    sigma=sigma,
    average='inner', #this extra keyword makes it a global descriptor by averaging over all atoms in the molecule
)


Now we have generated three sets of descriptors

- MBTRs with 2 body interactions (distances only)
- SOAPs with 3 body interactions (low cutoff and angular momentum)
- SOAPS with 3 body interactions (higher cutoff and angular momentum)

## 3. Dimensionality Reduction

Dimensionality reduction also sometimes goes under the name of "finding a low-dimensional embedding" or "manifold learning". 

### Principal Component Analysis (PCA)

PCA is the simplest linear dimensionality reduction technique based on Singular value Decomposition. The idea is to find a small set of principal components that represent the maximum amount of variance (diversity) in the data. We then project the data into that lower dimensional space. There are many variations on this technique including non-linear ones (Kernel PCA).

In [ ]:
from sklearn.decomposition import PCA

#pick either 
descriptors = soaps_default
#descriptors = mbtrs
#descriptors = soaps_high

n_components = 10

pca = PCA(n_components=n_components) #looking at the 5 largest eigenvalues
pca.fit(descriptors)
# this is the part where we transform the data into the PCA representation. Now each element of the descriptor corresponds to the respective PCA eigenvalue
X_new = pca.transform(descriptors) 

# --- Scree plot ---
components = np.arange(1, n_components + 1)

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(components, pca.explained_variance_ratio_, color='steelblue', alpha=0.7)
ax.plot(components, pca.explained_variance_ratio_, marker='o', color='steelblue')
ax.set_xlabel('Principal Component')
ax.set_ylabel('Proportion of Variance Explained')
ax.set_xticks(components)
plt.title('Scree Plot')
plt.tight_layout()
plt.show()

print('Total variance of {0} components: {1:.3f}'.format(n_components, pca.explained_variance_ratio_.sum()))
print('Variance of first 2 components: {0:.3f}'.format(pca.explained_variance_ratio_[:2].sum()))

The above plot shows us the variance captured by the largest eigenvalues. Together the components dominantly represent the data. The 2 lowest components cover a significant part of the variance.

**Task for you**
<div class="alert alert-block alert-info">
    
- Judging by the Scree plot, report how many principal components should be included in the dimensionality reduction for each of the three descriptors
- Report the length of each of the descriptor vectors
- Which descriptor gives the highest variance for the first two PCA components?
</div>

In [ ]:
# your answer here

We pick the two PCs with the highest variance over the data and plot them against each other

In [ ]:
print(X_new.shape)
descriptors = X_new[:,:2]
print(descriptors.shape)

In [ ]:
fig, ax = plt.subplots(1, 1)

for (r, rgb) in zip(ranges,rgb_colors):
    ax.scatter(descriptors[r[0]:r[1],0], descriptors[r[0]:r[1],1], edgecolors='white', color=rgb, linewidths=0.4, label="datapoints", alpha=0.8)

plt.xlabel("PC1")
plt.ylabel("PC2")
fig.set_figheight(6.0)
fig.set_figwidth(6.0)

plt.show()

Let's only plot the 'planar' trajectory (the purple one) and colour the dots according to the trajectory time.
(The planar trajectory constitutes the final 10000 data points)
Red represent early times, blue represent late times.

In [ ]:
fig, ax = plt.subplots(1, 1)

cm = plt.get_cmap('RdYlBu')
colors = range(10000)

sc = ax.scatter(descriptors[40000:,0], descriptors[40000:,1], edgecolors='white', 
            c=colors, linewidths=0.4, label="datapoints", alpha=0.8, cmap=cm)
fig.colorbar(sc)


plt.xlabel("PC1")
plt.ylabel("PC2")
fig.set_figheight(6.0)
fig.set_figwidth(6.0)

plt.show()

We can see that the deformation of the molecule out of the planar geometry can be captured well with the two principal components. However, the subtle differences between structures are not well resolved. We will try another method below.

**Task for you**
<div class="alert alert-block alert-info">
    
- Compare the PCA plots for MBTR (`mbtrs`), SOAP(n_max=4, l_max=4) (`soaps_default`), SOAP(n_max=6,l_max=6) (`soaps_high`) and create them side by side below. 
- Note the domains on the x and y axes covered by the datapoints for the different descriptors
- Which one best resolves the different conformations and the transition of the planar trajectory?

</div>

### t-SNE

 t-Distributed Stochastic Neighbor Embedding (t-SNE) is a popular dimensionality-reduction algorithm for visualizing high-dimensional data sets. It can be very informative (and sometimes also a bit misleading). Both methods are nonlinear (in contrast to PCA). t-SNE focuses on preserving the pairwise similarities between data points in a lower-dimensional space. t-SNE is concerned with preserving small pairwise distances whereas, PCA focuses on maintaining large pairwise distances to maximize variance. 
 
 So PCA preserves the variance in the data, whereas t-SNE preserves the relationships between data points in a lower-dimensional space, making it quite a good algorithm for visualizing complex high-dimensional data and relationships between individual datapoints. 

 Popular libraries for t-SNE (or an alternative algorithm called UMAP) are
 - [openTSNE](https://opentsne.readthedocs.io/en/stable/)
 - [umap-learn](https://umap-learn.readthedocs.io/en/latest/)

In [ ]:
from openTSNE import TSNE

In [ ]:
#list of colors where each point is colored according to its trajectory

subsampling = 10 #consider only every 10th datapoint

data_length = 10000//subsampling

colors = []
for c in rgb_colors:
    for i in range(data_length):
        colors.append(c)

#list of colors for one trajectory which indicates time


##pick a descriptor
descriptors = soaps_low[::subsampling,:]
#descriptors = mbtrs[::subsampling,:]
#descriptors = soaps_high[::subsampling,:]


The **perplexity** is an important tunable parameter for t-SNE. 
Here we can see how increasing the perplexity (number of expected neighbors) changes the layout of the projection.

<div class="alert alert-block alert-danger">
    
This algorithm may take several minutes. Be sure to parallelize it (`n_jobs`) and be patient. Above, we have already sliced the datasets to only take every 10th datapoint (1000 data points for each trajectory).
</div>

In [ ]:
perplexities = np.logspace(0, 2.69, 6, dtype=int)
fig, ax = plt.subplots(1,
                       len(perplexities),
                       figsize=(4 * len(perplexities), 4),
                      )

for i, perp in enumerate(tqdm(perplexities)):
    tsne = TSNE(
        n_components=2,  # number of components to project across
        perplexity= perp, 
        metric="euclidean",  # distance metric
        n_jobs=6,  # parallelization
        random_state=42, #fixed random seed for reproducibility
        verbose=False,
    )
    t_tsne = tsne.fit(descriptors)
    ax[i].scatter(*t_tsne.T, c=colors, s=2)
    ax[i].axis('off')
    ax[i].set_title("Perplexity = {}".format(perp))
plt.show()

For MBTR and SOAP and for large enough perplexity values, we find that the data clusters into 2 very large clusters and 1 very little one. The colours we are using are still the colours associated with the five cyclohexane MD runs. The t-SNE is proof that the 5 conformers start in different places but equilibrate into only two structures, namely the `boat` (green, red, yellow trajectories) and the `chair` conformation (blue and purple trajectories). 



Now let's take the most suitable perplexity value and look at the t-SNE, but now we color it according to the trajectory time. Red points are early, purple points are towards the end of the trajectory. We represent the different trajectory types with different markers. FOcus on the circles (planar trajectory).

In [ ]:
perplexity = 50
plt.figure(figsize=(4,4),dpi=300)

cm = plt.cm.get_cmap('RdYlBu')
colors_time = []
for r in range(5):
    for i in np.linspace(0,1,data_length):
        colors_time.append(cm(i))

markers = []
for c in ['v','^','<','>','o']:
    for i in range(data_length):
        markers.append(c)

tsne = TSNE(
    n_components=2,  # number of components to project across
    perplexity=perplexity,  # amount of neighbors one point is posited to have... play around with this!
    metric="euclidean",  # distance metric
    n_jobs=4,  # parallelization
    random_state=42,
    verbose=False,
)

size = 4
t_tsne = tsne.fit(descriptors)
for i,m in enumerate(['v','^','<','>','o']):
    r = ranges[i]//subsampling
    tmp = t_tsne[r[0]:r[1]]
    plt.scatter(*tmp.T, c=colors_time[r[0]:r[1]], marker=m, s=size)

plt.axis('off')
plt.title("Perplexity = {}".format(perplexity))
plt.show()

The planar trajectory (circles) starts off in a place that is structurally similar to the initial configurations of the `boat` (yellow previousy, '>' signs here) trajectory. While the `boat` trajectory converges into the same space as the half-chair and twist-boat, the `planar` trajectory moves through the `half-chair` configuration and the `boat` configuration before it ends up in the `chair` configuration.

**Task for you**:
<div class="alert alert-block alert-info">

- Try out different distance metrics in the t-SNR plots

- A big part of ML and data science is data visualization. Can you come up with a better way to visualize the dynamical changes of conformations along the trajectories based on what we know now?
</div>

In [ ]:
#space to play around

## 4. Clustering

Let's assume we use PCA or t-SNE to perform dimensionality reduction and in the process, we have found a way to visualize the landscape of the data and relevant similarities. If we trust the space, we might want to perform clustering to identify the $n$ most structurally diverse geometries from this space. 

This would allow us for example to pick a set of structures that best represent the diversity of this space to perform subsequent electronic structure calculations.

We will use [k-means clustering](https://scikit-learn.org/stable/modules/clustering.html#k-means) based on the previous PCA to achieve this. We will use the computationally more efficient batched variant of KMeans.

In [ ]:
from sklearn.decomposition import PCA

#pick either 
descriptors = soaps_default
#descriptors = mbtrs
#descriptors = soaps_high


###regenerate PCA
n_components = 5
pca = PCA(n_components=n_components)
pca.fit(descriptors)
X_new = pca.transform(descriptors) 
print(X_new.shape)
descriptors = X_new[:,:2]

In [ ]:
from sklearn.cluster import MiniBatchKMeans

niter = 100000
minibatch = 1800 # Batch size (how many data points are treated in the algorithm at once, reduce if you run out of memory)
ncluster= 5 # number of clusters

kmeans = MiniBatchKMeans(n_clusters=ncluster,
                        init="k-means++",
                        max_iter=niter,
                        n_init=3,
                        batch_size=minibatch)
# Fit the clustering model
km = kmeans.fit(np.array(descriptors).reshape(-1,1))
# Use the clustering model to predict the cluster number of each data point
indices = km.fit_predict(descriptors)

Once we have performed the clustering, we can calculate the centres of clusters, that is, the indices of the datapoints that are closest to the centroids of the clusters.

In [ ]:
from sklearn.metrics.pairwise import pairwise_distances_argmin

centroid = kmeans.cluster_centers_
b = np.inf
ind = pairwise_distances_argmin(centroid, descriptors) # measure which structure is closest to the centroids of the clusters
print(ind)

X_centers = []
for i in ind:
    X_centers.append(descriptors[i])
X_centers = np.array(X_centers)

Now you could save the indices and you can identify which conformations belong to thse clusters. Let's look at them

In [ ]:
mol_traj = []
for i in ind:
    mol_traj.append(traj[i])

# visualize the trajectory of the clustered molecules
viewer = WeasWidget()
viewer.from_ase(mol_traj)
viewer.avr.model_style = 1
viewer.avr.show_hydrogen_bonds = True
viewer

Now we replot the PCA plot while coloring the different clusters and highlighting their centres.

In [ ]:
fig, ax = plt.subplots(1, 1)

ax.scatter(X_new[:,0], X_new[:,1],c=indices, edgecolors='white', linewidths=0.4, label="datapoints", alpha=0.8)
ax.scatter(X_centers[:,0],X_centers[:,1],color="white",label="centers",edgecolors='black',linewidths=0.7,marker="X", alpha=0.8)

plt.legend(fancybox=True,framealpha=1,edgecolor='black',handletextpad=0.05,borderpad=0.3,handlelength=1.2,columnspacing=0.4,labelspacing=0.2,ncol=1,loc=1, fontsize="medium") #, bbox_to_anchor=(1.75, 1.02))
plt.xlabel("PC1")
plt.ylabel("PC2")
fig.set_figheight(6.0)
fig.set_figwidth(6.0)

plt.show()
# Two distinct parts of PC1/2 space, so pick enough clusters to avoid clusters spanning both parts. 


We can also see if our clusters cover a wide range of values for PC1

In [ ]:
from matplotlib.ticker import MaxNLocator, FuncFormatter

ax = plt.axes()

ax.scatter(X_new[:,0], indices, edgecolors='white', linewidths=0.4, alpha=0.8)

plt.xlabel("PC1")
plt.ylabel("Cluster")
ax.xaxis.set_major_locator(MaxNLocator(integer=True))
ax.yaxis.set_major_locator(MaxNLocator(integer=True))

plt.show()

**Task for you**:
<div class="alert alert-block alert-info">
    
- Try playing around with the number of clusters. How big a number would still be a sensible set of clusters?

- Clearly K-means clustering is not ideal for this dataset as 3 trajectories fall into one minimum and 2 fall into another one. Try DBSCAN and HDBSCAN. Find optimal parameters and plot the clustering results

</div>

## 5. Homework Assignments

**Task for you**:
<div class="alert alert-block alert-info">
    
- Generate descriptors for the QM7 dataset that nicely resolve similarities between molecules and capture 90% of variance in 2 PCs
- Generate 2D PCA and optimally-tunded t-SNE plots with all molecules in QM7.
- Does the dataset show clusters or emerging groupings of molecules? 
- Use clustering (test a few algorithms and then pick the appropriate algorithm) to identify the 10 most diverse molecules in the dataset (according to the PCs)
- Visualise the 10 molecules and analyse what structural features differentiate them

</div>

In [ ]:
#code here